# Experimento A: Establecimiento de la Línea Base (Baseline Puro)

## Objetivo
Medir el rendimiento **"natural"** de los algoritmos sin ninguna ayuda para el desbalance.  
Esto sirve de **punto de referencia (suelo)** para cuantificar la mejora real de las técnicas posteriores.

## Modelos
- **Regresión Logística**: Modelo lineal simple de referencia
- **Random Forest (RF)**: Ensamble robusto estándar
- **XGBoost**: Estado del arte en boosting

## Metodología (Capítulo 5)
- **Validación prequential:** 4 folds temporales
- **GridSearchCV** con búsqueda de hiperparámetros por modelo
- **Métricas:** AUC ROC, AUPRC, Card Precision@100 (media ± desv. estándar)
- **Sin** `class_weight`, **sin** SMOTE
- Preprocesamiento: `StandardScaler` en features numéricas

## Hipótesis
Se espera una Accuracy altísima (~99%) pero un Recall de fraude muy bajo (<20-30%),  
demostrando la inutilidad de la métrica Accuracy en problemas desbalanceados.

## Métrica Clave
- AUPRC (Area Under Precision-Recall Curve)
- Card Precision@100

---
## 1. Imports y Configuración

In [ ]:
import os
import sys
import time
import datetime
import warnings
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.metrics

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import metrics
import xgboost as xgb

# Configuración del proyecto
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, INPUT_FEATURES, OUTPUT_FEATURE,
    RESULTS_DIR, FIGURES_DIR,
    TOP_K_LIST, COLORS,
    DELTA_TRAIN, DELTA_DELAY, DELTA_TEST,
    START_DATE_TRAINING_FOR_VALID, START_DATE_TRAINING_FOR_TEST,
    N_FOLDS,
)
from experiments.data_utils import (
    load_transformed_data,
    card_precision_top_k, card_precision_top_k_custom,
    model_selection_wrapper,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})

# Modo rápido: 2 folds y grid reducido (~3-5 min). False = completo (4 folds, ~15-30 min)
QUICK_VALIDATION = False  # True para validación rápida; False para resultados finales

n_folds = 2 if QUICK_VALIDATION else N_FOLDS
print(f"Semilla de reproducibilidad: {SEED}")
print(f"Features de entrada: {len(INPUT_FEATURES)}")
print(f"Variable objetivo: {OUTPUT_FEATURE}")
print(f"Validación prequential: {n_folds} folds {'(modo rápido)' if QUICK_VALIDATION else ''}")

---
## 2. Carga de Datos

In [ ]:
# Cargar dataset transformado (con feature engineering del Chapter 3)
transactions_df = load_transformed_data()
print(f"Dataset cargado: {len(transactions_df):,} transacciones")
print(f"Periodo: {transactions_df.TX_DATETIME.min()} → {transactions_df.TX_DATETIME.max()}")
print(f"Fraude: {transactions_df[OUTPUT_FEATURE].sum():,} transacciones ({100*transactions_df[OUTPUT_FEATURE].mean():.2f}%)")

In [ ]:
# Scorer personalizado para Card Precision@100 (compatible con GridSearchCV)
card_precision_scorer = sklearn.metrics.make_scorer(
    card_precision_top_k_custom,
    needs_proba=True,
    top_k=100,
    transactions_df=transactions_df,
)

# Scoring: AUC ROC, AUPRC, CP@100
scoring = {
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
    'card_precision@100': card_precision_scorer,
}

performance_metrics_list_grid = ['roc_auc', 'average_precision', 'card_precision@100']
performance_metrics_list = ['AUC ROC', 'Average precision', 'Card Precision@100']

# Grids de hiperparámetros (metodología Capítulo 5, sin class_weight/scale_pos_weight)
if QUICK_VALIDATION:
    param_grids = {
        "Logistic Regression": {'clf__C': [0.1, 1, 10], 'clf__max_iter': [1000], 'clf__random_state': [SEED]},
        "Random Forest": {'clf__max_depth': [10, 50], 'clf__n_estimators': [50], 'clf__random_state': [SEED], 'clf__n_jobs': [-1]},
        "XGBoost": {'clf__max_depth': [3, 6], 'clf__n_estimators': [50], 'clf__learning_rate': [0.3], 'clf__random_state': [SEED],
                    'clf__use_label_encoder': [False], 'clf__eval_metric': ['logloss'], 'clf__n_jobs': [-1], 'clf__verbosity': [0]},
    }
else:
    param_grids = {
        "Logistic Regression": {'clf__C': [0.1, 1, 10, 100], 'clf__max_iter': [1000], 'clf__random_state': [SEED]},
        "Random Forest": {'clf__max_depth': [10, 20, 50], 'clf__n_estimators': [50, 100], 'clf__random_state': [SEED], 'clf__n_jobs': [-1]},
        "XGBoost": {'clf__max_depth': [3, 6, 9], 'clf__n_estimators': [50, 100], 'clf__learning_rate': [0.3],
                   'clf__random_state': [SEED], 'clf__use_label_encoder': [False], 'clf__eval_metric': ['logloss'],
                   'clf__n_jobs': [-1], 'clf__verbosity': [0]},
    }

---
## 3. Entrenamiento y Selección de Modelos (Prequential + GridSearch)

Validación prequential (4 folds) + búsqueda de hiperparámetros por modelo.  
**No se aplica ninguna técnica para manejar el desbalance** (sin class_weight, sin SMOTE).

In [ ]:
# Clasificadores baseline (sin class_weight / scale_pos_weight)
classifiers = {
    "Logistic Regression": LogisticRegression(),
    "Random Forest": RandomForestClassifier(),
    "XGBoost": xgb.XGBClassifier(),
}

print("=" * 70)
print("  EXPERIMENTO A: BASELINE PURO — Validación Prequential + GridSearch")
print("=" * 70)

results_a = {}
start_total = time.time()

for name, clf in classifiers.items():
    print(f"\n{'─' * 50}")
    print(f"  Procesando: {name}")
    print(f"  Grid: {len(param_grids[name])} parámetros → múltiples combinaciones")
    start_model = time.time()

    performances_df = model_selection_wrapper(
        transactions_df, clf,
        INPUT_FEATURES, OUTPUT_FEATURE,
        param_grids[name], scoring,
        start_date_training_for_valid=START_DATE_TRAINING_FOR_VALID,
        start_date_training_for_test=START_DATE_TRAINING_FOR_TEST,
        n_folds=n_folds,
        delta_train=DELTA_TRAIN,
        delta_delay=DELTA_DELAY,
        delta_assessment=DELTA_TEST,
        performance_metrics_list_grid=performance_metrics_list_grid,
        performance_metrics_list=performance_metrics_list,
        n_jobs=-1,
    )

    # Mejor config por AUPRC en validación
    best_idx = performances_df['Average precision Validation'].idxmax()
    best_row = performances_df.loc[best_idx]

    results_a[name] = {
        'performances_df': performances_df,
        'best_params': performances_df.loc[best_idx, 'Parameters'],
        'auc_roc_mean': best_row['AUC ROC Test'],
        'auc_roc_std': best_row['AUC ROC Test Std'],
        'auprc_mean': best_row['Average precision Test'],
        'auprc_std': best_row['Average precision Test Std'],
        'cp100_mean': best_row['Card Precision@100 Test'],
        'cp100_std': best_row['Card Precision@100 Test Std'],
    }

    elapsed = time.time() - start_model
    print(f"  ✓ Completado en {elapsed:.1f}s")
    print(f"    AUC ROC: {results_a[name]['auc_roc_mean']:.4f} ± {results_a[name]['auc_roc_std']:.4f}")
    print(f"    AUPRC:   {results_a[name]['auprc_mean']:.4f} ± {results_a[name]['auprc_std']:.4f}")
    print(f"    CP@100:  {results_a[name]['cp100_mean']:.4f} ± {results_a[name]['cp100_std']:.4f}")

print(f"\n{'=' * 70}")
print(f"  Tiempo total: {time.time() - start_total:.1f}s")
print("=" * 70)

In [ ]:
# Resumen de mejores hiperparámetros (por AUPRC en validación)
for name, res in results_a.items():
    print(f"{name}: {res['best_params']}")

---
## 4. Tabla Comparativa y Visualizaciones

In [ ]:
# Tabla de resultados (media ± desv. estándar sobre 4 folds prequential)
model_names = list(results_a.keys())
results_table = pd.DataFrame({
    'Modelo': model_names,
    'AUC ROC': [f"{results_a[n]['auc_roc_mean']:.4f} ± {results_a[n]['auc_roc_std']:.4f}" for n in model_names],
    'AUPRC': [f"{results_a[n]['auprc_mean']:.4f} ± {results_a[n]['auprc_std']:.4f}" for n in model_names],
    'CP@100': [f"{results_a[n]['cp100_mean']:.4f} ± {results_a[n]['cp100_std']:.4f}" for n in model_names],
}).set_index('Modelo')

# Versión numérica para exportar (solo medias)
results_table_numeric = pd.DataFrame({
    'Modelo': model_names,
    'AUC ROC': [results_a[n]['auc_roc_mean'] for n in model_names],
    'AUC ROC Std': [results_a[n]['auc_roc_std'] for n in model_names],
    'AUPRC': [results_a[n]['auprc_mean'] for n in model_names],
    'AUPRC Std': [results_a[n]['auprc_std'] for n in model_names],
    'CP@100': [results_a[n]['cp100_mean'] for n in model_names],
    'CP@100 Std': [results_a[n]['cp100_std'] for n in model_names],
}).set_index('Modelo')

print("\nTabla comparativa - Experimento A (Baseline Puro, validación prequential):")
print("=" * 80)
display(results_table)

In [ ]:
# Gráfico de barras: AUC ROC, AUPRC y CP@100 (media ± desv. estándar)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

model_names = list(results_a.keys())
x_pos = np.arange(len(model_names))
width = 0.5

metrics_to_plot = [
    ('auc_roc', 'AUC ROC', COLORS['baseline']),
    ('auprc', 'AUPRC', COLORS['correct_pipeline']),
    ('cp100', 'Card Precision@100', COLORS['cost_sensitive']),
]

for ax, (key, label, color) in zip(axes, metrics_to_plot):
    means = [results_a[n][f'{key}_mean'] for n in model_names]
    stds = [results_a[n][f'{key}_std'] for n in model_names]
    bars = ax.bar(x_pos, means, width, yerr=stds, capsize=5, color=color, edgecolor='black')
    ax.set_ylabel(label, fontsize=12)
    ax.set_title(f'{label}\n(Experimento A — 4 folds prequential)', fontsize=12)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(model_names, rotation=15, ha='right')
    ax.set_ylim([0, 1.05])
    for bar, m, s in zip(bars, means, stds):
        ax.annotate(f'{m:.3f}±{s:.3f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_a_baseline_results.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nFigura guardada en: {FIGURES_DIR / 'experiment_a_baseline_results.png'}")

---
## 5. Guardar Resultados

In [ ]:
# Guardar tabla de resultados (numérica con medias y desviaciones)
results_table_numeric.to_csv(RESULTS_DIR / 'experiment_a_results.csv')

# Guardar resultados completos (performances_df, best_params, métricas) para comparar con Exp B y C
# Claves compatibles con experiment_c: card_precision_at_k, avg_precision, auc_roc
results_to_save = {
    name: {
        'auc_roc_mean': res['auc_roc_mean'],
        'auc_roc_std': res['auc_roc_std'],
        'auprc_mean': res['auprc_mean'],
        'auprc_std': res['auprc_std'],
        'cp100_mean': res['cp100_mean'],
        'cp100_std': res['cp100_std'],
        'best_params': res['best_params'],
        'performances_df': res['performances_df'],
        # Compatibilidad con experiment_c_leakage_test
        'card_precision_at_k': {100: res['cp100_mean']},
        'avg_precision': res['auprc_mean'],
        'auc_roc': res['auc_roc_mean'],
    }
    for name, res in results_a.items()
}

with open(RESULTS_DIR / 'experiment_a_predictions.pkl', 'wb') as f:
    pickle.dump(results_to_save, f)

print("✓ Resultados del Experimento A guardados exitosamente")
print(f"  - CSV: {RESULTS_DIR / 'experiment_a_results.csv'}")
print(f"  - PKL: {RESULTS_DIR / 'experiment_a_predictions.pkl'}")

---
## 6. Conclusiones del Experimento A

**Validación de la hipótesis:**

- La **Accuracy** es engañosamente alta (~99%) porque el modelo simplemente predice "no fraude" la mayoría del tiempo.
- El **Recall** de la clase fraude es muy bajo, lo que significa que el modelo falla en detectar los casos que realmente importan.
- La **AUPRC** es la métrica que mejor refleja el rendimiento real en este contexto desbalanceado.

Estos resultados establecen el **suelo de rendimiento** contra el cual se compararán los Experimentos B, C y D.